# LAB 3: Docling — Tabelas e Formulários Jurídicos
## Aula 5 — Docling e Ingestão Inteligente de Documentos
### MBA em RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**⏱️ Duração estimada:** 50 minutos  
**📚 Pré-requisitos:** Labs 1 e 2 concluídos  
**🎯 Objetivos:**
- Extrair tabelas estruturadas de laudos e relatórios com Docling
- Converter tabelas para DataFrames pandas para análise
- Processar formulários de BO com campos estruturados
- Implementar chunking especial que preserva integridade de tabelas
- Criar representações textuais de tabelas para indexação vetorial

In [ ]:
!pip install docling>=2.0.0 reportlab pandas --quiet
print("✅ Instalação concluída!")

In [ ]:
import sys, warnings, os, json, re
from pathlib import Path
import pandas as pd
warnings.filterwarnings('ignore')

from docling.document_converter import DocumentConverter
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors

print("✅ Ambiente configurado")

## 📄 Etapa 1: Criar Relatório de Inteligência com Tabelas Complexas

Vamos criar um relatório de inteligência policial com múltiplas tabelas
para testar as capacidades de extração do Docling.

**⏱️ Tempo estimado: 8 minutos**

In [ ]:
CAMINHO_RELATORIO = '/tmp/relatorio_inteligencia.pdf'

doc = SimpleDocTemplate(CAMINHO_RELATORIO, pagesize=A4,
    rightMargin=2*cm, leftMargin=2*cm, topMargin=2*cm, bottomMargin=2*cm)

styles = getSampleStyleSheet()
st_titulo = ParagraphStyle('t', parent=styles['Heading1'], fontSize=13, alignment=TA_CENTER)
st_h2 = ParagraphStyle('h2', parent=styles['Heading2'], fontSize=11, spaceAfter=6)
st_normal = ParagraphStyle('n', parent=styles['Normal'], fontSize=9, spaceAfter=4)

elementos = []
elementos.append(Paragraph('RELATÓRIO DE INTELIGÊNCIA POLICIAL', st_titulo))
elementos.append(Paragraph('Nº REL-DENARC-2024-0891 | CLASSIFICAÇÃO: RESTRITO', st_normal))
elementos.append(Spacer(1, 0.5*cm))

# Tabela 1: Estrutura da organização criminosa
elementos.append(Paragraph('1. ESTRUTURA DA ORGANIZAÇÃO', st_h2))

dados_org = [
    ['Nível', 'Função', 'Qtd. Membros', 'Status Investigação'],
    ['1 - Liderança', 'Contato fornecedores / Controle financeiro', '3', 'Identificados'],
    ['2 - Gerência', 'Distribuição regional / Lavagem', '8', 'Identificados'],
    ['3 - Operacional', 'Transporte e distribuição', '42', 'Parcialmente identificados'],
    ['TOTAL', '', '53', '11 alvos prioritários'],
]

tab_org = Table(dados_org, colWidths=[3.5*cm, 6*cm, 2.5*cm, 4*cm])
tab_org.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.darkgrey),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 8),
    ('ALIGN', (0,0), (-1,-1), 'CENTER'),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
    ('GRID', (0,0), (-1,-1), 0.5, colors.black),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.lightgrey, colors.white]),
]))
elementos.append(tab_org)
elementos.append(Spacer(1, 0.5*cm))

# Tabela 2: Movimentação financeira
elementos.append(Paragraph('2. MOVIMENTAÇÃO FINANCEIRA (2023)', st_h2))

dados_fin = [
    ['Categoria', 'Valor Estimado', 'Fonte de Informação'],
    ['Receita total estimada', 'R$ 4.700.000,00', 'Análise de interceptações'],
    ['Valor apreendido (espécie)', 'R$ 387.000,00', 'Operações anteriores'],
    ['Ativos imobiliários identificados', 'R$ 1.200.000,00', 'COAF / Cartório'],
    ['Veículos identificados', 'R$ 340.000,00', 'DETRAN cruzamento'],
    ['Valor não localizado', 'R$ 2.773.000,00', 'Em investigação'],
]

tab_fin = Table(dados_fin, colWidths=[6*cm, 4*cm, 6*cm])
tab_fin.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.navy),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 8),
    ('ALIGN', (1,1), (1,-1), 'RIGHT'),
    ('GRID', (0,0), (-1,-1), 0.5, colors.black),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.lightyellow, colors.white]),
]))
elementos.append(tab_fin)
elementos.append(Spacer(1, 0.5*cm))

# Tabela 3: Alvos prioritários
elementos.append(Paragraph('3. ALVOS PRIORITÁRIOS', st_h2))

dados_alvos = [
    ['ID', 'Codinome', 'Nível', 'Mandado', 'Observações'],
    ['A01', 'COBRA', '1', 'Prisão + Busca', 'Líder da organização'],
    ['A02', 'ESCUDO', '1', 'Prisão + Busca', 'Controlador financeiro'],
    ['A03', 'NEXO', '2', 'Prisão + Busca', 'Gerente região Norte'],
    ['A04', 'SOMBRA', '2', 'Busca', 'Ponto de distribuição'],
    ['A05', 'PRETO', '2', 'Prisão', 'Transportador principal'],
]

tab_alvos = Table(dados_alvos, colWidths=[1.5*cm, 3*cm, 2*cm, 4*cm, 5.5*cm])
tab_alvos.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.darkred),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
    ('FONTSIZE', (0,0), (-1,-1), 8),
    ('GRID', (0,0), (-1,-1), 0.5, colors.black),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.lightpink, colors.white]),
]))
elementos.append(tab_alvos)

doc.build(elementos)
print(f"✅ Relatório criado: {CAMINHO_RELATORIO}")
print(f"   Tamanho: {os.path.getsize(CAMINHO_RELATORIO)/1024:.1f} KB")

## 🔬 Etapa 2: Extrair Tabelas com Docling

Agora vamos usar o Docling para extrair as tabelas do relatório.

**⏱️ Tempo estimado: 12 minutos**

In [ ]:
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions

# Pipeline com detecção de tabelas habilitada
pipeline = PdfPipelineOptions(do_table_structure=True)
converter = DocumentConverter()

print("⏳ Processando relatório...")
resultado = converter.convert(CAMINHO_RELATORIO)
doc = resultado.document

print(f"\n✅ Processamento concluído")
print(f"   • Tabelas detectadas: {len(doc.tables)}")
print(f"   • Páginas: {len(doc.pages)}")

In [ ]:
# Extrair e analisar cada tabela
from docling_core.types.doc import TableItem

def extrair_tabelas_para_df(doc):
    # Extrai todas as tabelas do DoclingDocument como DataFrames
    tabelas = []
    for i, tabela in enumerate(doc.tables):
        info = {
            'indice': i,
            'linhas': tabela.data.num_rows if tabela.data else 0,
            'colunas': tabela.data.num_cols if tabela.data else 0,
            'markdown': tabela.export_to_markdown(doc),
            'dataframe': None
        }

        # Tentar converter para DataFrame
        try:
            if tabela.data and tabela.data.grid:
                dados = []
                for linha in tabela.data.grid:
                    dados.append([celula.text.strip() for celula in linha])
                if len(dados) > 1:
                    df = pd.DataFrame(dados[1:], columns=dados[0])
                    info['dataframe'] = df
        except Exception as e:
            print(f'Aviso: nao foi possivel converter tabela {i} para DataFrame: {e}')

        tabelas.append(info)
    return tabelas

tabelas_extraidas = extrair_tabelas_para_df(doc)
print(f"\n📊 {len(tabelas_extraidas)} tabela(s) extraída(s)\n")

for tab in tabelas_extraidas:
    print(f"=== Tabela {tab['indice']+1} ({tab['linhas']}x{tab['colunas']}) ===")
    print(f"Markdown:\n{tab['markdown'][:400]}")
    if tab['dataframe'] is not None:
        print(f"\nDataFrame:\n{tab['dataframe'].to_string()}")
    print()

## 📝 Etapa 3: Representação Textual de Tabelas para RAG

Tabelas em formato Markdown são adequadas para leitura humana, mas para
indexação vetorial, frequentemente é melhor converter para texto natural.

**⏱️ Tempo estimado: 10 minutos**

In [ ]:
def tabela_para_texto_natural(tabela_markdown, contexto=''):
    # Converte tabela Markdown para texto natural descritivo para RAG
    linhas = [l.strip() for l in tabela_markdown.strip().split('\n') if l.strip()]
    if len(linhas) < 2:
        return tabela_markdown

    # Extrair cabeçalho
    cabecalho = [c.strip() for c in linhas[0].split('|') if c.strip()]

    # Pular linha separadora (|---|---|)
    dados_inicio = 2 if len(linhas) > 2 and '---' in linhas[1] else 1

    # Converter cada linha em sentença
    sentencas = []
    if contexto:
        sentencas.append(f'TABELA - {contexto}:')

    for linha_raw in linhas[dados_inicio:]:
        celulas = [c.strip() for c in linha_raw.split('|') if c.strip()]
        if len(celulas) != len(cabecalho):
            continue

        # Montar sentença: 'Coluna1: Valor1; Coluna2: Valor2; ...'
        partes = [f'{col}: {val}' for col, val in zip(cabecalho, celulas)]
        sentencas.append(' | '.join(partes) + '.')

    return '\n'.join(sentencas)

# Aplicar nas tabelas extraídas
print('=== TABELAS CONVERTIDAS PARA TEXTO NATURAL ===')
contextos = [
    'Estrutura da organização criminosa (Operação NEXUS)',
    'Movimentação financeira estimada - 2023',
    'Alvos prioritários para mandados de prisão'
]

textos_tabelas = []
for i, tab in enumerate(tabelas_extraidas):
    ctx = contextos[i] if i < len(contextos) else f'Tabela {i+1}'
    texto = tabela_para_texto_natural(tab['markdown'], ctx)
    textos_tabelas.append(texto)
    print(f'\n--- {ctx} ---')
    print(texto[:500])

## ✅ Checkpoint — Lab 3

Execute a célula abaixo para verificar os objetivos atingidos.

In [ ]:
print("=" * 60)
print("CHECKPOINT — LAB 3")
print("=" * 60)
erros = []

if not Path(CAMINHO_RELATORIO).exists():
    erros.append('❌ Relatório PDF não criado')
else:
    print("✅ Relatório com tabelas criado")

if len(tabelas_extraidas) == 0:
    erros.append('❌ Nenhuma tabela extraída')
else:
    print(f"✅ {len(tabelas_extraidas)} tabela(s) extraída(s)")

dfs_ok = sum(1 for t in tabelas_extraidas if t['dataframe'] is not None)
if dfs_ok == 0:
    erros.append('❌ Nenhuma tabela convertida para DataFrame')
else:
    print(f"✅ {dfs_ok} tabela(s) convertida(s) para DataFrame")

if len(textos_tabelas) == 0:
    erros.append('❌ Conversão para texto natural falhou')
else:
    print(f"✅ {len(textos_tabelas)} tabela(s) convertida(s) para texto")

if erros:
    for e in erros: print(e)
else:
    print("\n🎉 TODOS OS OBJETIVOS ATINGIDOS! Avance para o Lab 4.")

## 🏋️ Exercício

**Desafio:** Adicione uma 4ª tabela ao relatório (ex: tabela de veículos
apreendidos com colunas: Modelo, Ano, Placa, Valor) e verifique se o
Docling consegue extraí-la corretamente.

Dica: adicione antes de `doc.build(elementos)` no código da Etapa 1.

## 📖 Referências (ABNT)

AUER, Peter et al. **Docling Technical Report**. arXiv:2408.09869, 2024.

PRASAD, Vijay; SAHA, Rajat. **Table Detection and Structure Recognition: Survey**.
arXiv:2211.08469, 2022.
